In [ ]:
import os, re
import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

TOTAL_TOKENS = 3_001_757_696

# ========= 在这里填入你要绘制的文件夹列表 =========
FOLDERS = [
    "results_pretrain/base_rope_42",
    "results_pretrain/base_nope_w64s0g0h4g1_42",
]
# ==================================================


def name_to_label(name):
    """文件夹名 -> 图例中的模型名：按 _ 分隔，去掉第一个和最后一个，再用空格连接。"""
    parts = name.split("_")
    name = " ".join(parts[0:-1])
    return {
        "base rope": "RoPE 124.7M baseline",
        "base nope w64s0g0h4g1": "ConvPE 125.3M",
    }.get(name, name)


def read_trainer_state(ckpt_dir):
    path = os.path.join(ckpt_dir, "trainer_state.json")
    if not os.path.isfile(path):
        return None
    with open(path) as f:
        return json.load(f)


fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

model_handles, model_labels = [], []

for idx, folder in enumerate(FOLDERS):
    color = colors[idx % len(colors)]
    folder = folder.rstrip("/")
    name = os.path.basename(folder)
    label = name_to_label(name)
    if not os.path.isdir(folder):
        print(f"[WARN] 文件夹不存在: {folder}")
        continue

    # 找到最新的 checkpoint 目录（按编号排序）
    ckpts = sorted(
        (int(m.group(1)), d)
        for d in os.listdir(folder)
        if (m := re.fullmatch(r"checkpoint-(\d+)", d))
        and os.path.isdir(os.path.join(folder, d))
    )
    if not ckpts:
        continue

    state = read_trainer_state(os.path.join(folder, ckpts[-1][1]))
    if not state:
        continue

    train_pts = sorted(
        (e["epoch"], e["loss"])
        for e in state.get("log_history", [])
        if "loss" in e and "eval_loss" not in e and "epoch" in e
    )
    if not train_pts:
        continue

    ax.plot(
        [p[0] for p in train_pts],
        [p[1] for p in train_pts],
        lw=2.0,
        alpha=0.85,
        color=color,
        linestyle="-",
    )
    ax.plot(
        [p[0] for p in train_pts],
        [p[1] for p in train_pts],
        lw=2.0,
        alpha=0.85,
        color=color,
        linestyle="-",
    )
    model_handles.append(Line2D([0], [0], color=color, lw=2.0))
    model_labels.append(label)

ax.legend(
    handles=model_handles,
    labels=model_labels,
    loc="upper right",
    title="Model",
    fontsize=13,
    framealpha=0.9,
    labelspacing=0.35,
    handlelength=1.8,
    borderpad=0.5,
    title_fontsize=13,
)

ax.set_xlabel("Epochs")
ax.set_ylabel("Training Loss")
ax.set_title("Training Loss over the First 3B Tokens of FineWeb-Edu-10BT")
# ax.set_ylim(2.5, 5)
# ax.set_yscale("log")
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.3)
fig.tight_layout()

plt.show()